# Loading Tabular Data

The fastest way to get music data into TimeToAlign! is through **tabular loaders**. If your data is in CSV or TSV format, you're just 3 lines of code away from analysis.

**What you'll learn:**
- Load music annotations from TSV/CSV files
- Access event counts, coordinate ranges, and metadata
- Work with different coordinate types (seconds, beats, fractions)

**Time:** 5 minutes

## TL;DR

```python
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load("beethoven.notes.tsv")

print(f"{len(loader.events)} notes loaded")
```

## Setup

In [ ]:
from pathlib import Path

# Our specimen: Beethoven WoO 71 (a short piano piece)
SPECIMENS = Path(".").resolve().parents[1] / ".." / "dashboard" / "specimens"
BEETHOVEN = SPECIMENS / "beethoven_woo71"

# Available files
list(BEETHOVEN.glob("WoO71.*.tsv"))

## Loading Notes from TSV

The `Ms3Loader` handles TSV files exported from the [ms3](https://github.com/johentsch/ms3) parser, which processes MuseScore files.

**Three lines of code:**

In [ ]:
from timetoalign.loader.tabular import Ms3Loader

loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")

f"{len(loader.events):,} notes loaded"

## Quick Statistics

The loader provides immediate access to summary information:

In [ ]:
# What did we get?
{
    "event_count": len(loader.events),
    "coordinate_range": loader.events.coordinate_range(),
    "unit": str(loader.unit),
    "number_type": str(loader.number_type),
}

In [ ]:
# Event types (all notes in this file)
loader.count_events_by_type()

In [ ]:
# Temporal types: interval (has duration) vs instant (no duration)
loader.count_events_by_temporal_type()

## Understanding Coordinates

The coordinates tell us:
- **Range (0.0, 876.5):** The piece spans ~877 quarter beats
- **Unit: quarters:** Coordinates are in quarter note units
- **Number type: fraction:** Original data used fractions like "1/2", "3/4"

At 4 quarter beats per measure, 877 quarters = ~219 measures. Let's verify with the measures file:

In [ ]:
measures = Ms3Loader()
measures.load(BEETHOVEN / "WoO71.measures.tsv")

f"{len(measures.events)} measures"

## Loading Multiple Files

Load all notes files at once:

In [ ]:
all_notes = Ms3Loader()
all_notes.load(*BEETHOVEN.glob("*.notes.tsv"))

f"{len(all_notes.events):,} total notes from {len(list(BEETHOVEN.glob('*.notes.tsv')))} files"

## Performance

TabularLoaders are **vectorized** - they use numpy/pandas operations instead of Python loops. This means loading 10,000+ events takes milliseconds:

In [ ]:
import time

start = time.perf_counter()
loader = Ms3Loader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")
elapsed = time.perf_counter() - start

f"{len(loader.events):,} events in {elapsed*1000:.1f}ms = {len(loader.events)/elapsed:,.0f} events/sec"

## Generic CSV/TSV Loaders

For non-ms3 formats, use `CsvLoader` or `TsvLoader` with custom column mappings:

In [ ]:
from timetoalign.loader.tabular import CsvLoader, TsvLoader

# Default columns: start, end, id, name, event_type
# Customize via class attributes:

class MyLoader(CsvLoader):
    """Custom loader for my CSV format."""
    start_column = "onset_time"      # Column with start times
    end_column = "offset_time"       # Column with end times (optional)
    name_column = "note_name"        # Column with event names
    default_event_type = "Note"      # Default type for all events

# Now MyLoader() would parse CSVs with those column names

## Available Tabular Loaders

| Loader | Format | Use Case |
|--------|--------|----------|
| `Ms3Loader` | TSV | MuseScore exports via ms3 parser |
| `TsvLoader` | TSV | Generic tab-separated files |
| `CsvLoader` | CSV | Generic comma-separated files |
| `LabLoader` | LAB | Audacity/Praat label files |

## Summary

**Key takeaways:**

1. **3 lines to load:** Create loader, call `.load()`, access `.events`
2. **Instant statistics:** `.count_events_by_type()`, `.coordinate_range()`
3. **Fast:** 100,000+ events/second via vectorized operations
4. **Flexible:** Customize column mappings for any CSV/TSV format

**Next:** Learn about the full EventStore API and score-specific loaders in **02_loading_data.ipynb**

---

## Exercise

Load the Rachmaninoff Concerto 2 notes and answer:
1. How many notes are in the file?
2. What is the coordinate range?
3. How many interval vs instant events?

<details>
<summary>Hint</summary>

The file is at: `SPECIMENS / "rachmaninoff_concerto2" / "score" / "*.notes.tsv"`

</details>

In [ ]:
# Your solution here
